# Imports

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

# Dataset Path

In [2]:
LABELS_FOLDER = Path(
    r"D:\downloads\ToothFairy3\labelsTr"
)

print("Number of masks:", len(list(LABELS_FOLDER.glob("*.nii.gz"))))

Number of masks: 532


# Label Dictionary

In [3]:
LABELS = {
    1: "Lower Jawbone",
    2: "Upper Jawbone",
    3: "Left Inferior Alveolar Canal",
    4: "Right Inferior Alveolar Canal",
    5: "Left Maxillary Sinus",
    6: "Right Maxillary Sinus",
    7: "Pharynx",
    8: "Bridge",
    9: "Crown",
    10: "Implant",
    11: "Upper Right Central Incisor",
    12: "Upper Right Lateral Incisor",
    13: "Upper Right Canine",
    14: "Upper Right First Premolar",
    15: "Upper Right Second Premolar",
    16: "Upper Right First Molar",
    17: "Upper Right Second Molar",
    18: "Upper Right Third Molar",
    21: "Upper Left Central Incisor",
    22: "Upper Left Lateral Incisor",
    23: "Upper Left Canine",
    24: "Upper Left First Premolar",
    25: "Upper Left Second Premolar",
    26: "Upper Left First Molar",
    27: "Upper Left Second Molar",
    28: "Upper Left Third Molar",
    31: "Lower Left Central Incisor",
    32: "Lower Left Lateral Incisor",
    33: "Lower Left Canine",
    34: "Lower Left First Premolar",
    35: "Lower Left Second Premolar",
    36: "Lower Left First Molar",
    37: "Lower Left Second Molar",
    38: "Lower Left Third Molar",
    41: "Lower Right Central Incisor",
    42: "Lower Right Lateral Incisor",
    43: "Lower Right Canine",
    44: "Lower Right First Premolar",
    45: "Lower Right Second Premolar",
    46: "Lower Right First Molar",
    47: "Lower Right Second Molar",
    48: "Lower Right Third Molar",
    103: "Left Mandibular Incisive Canal",
    104: "Right Mandibular Incisive Canal",
    105: "Lingual Canal",
    111: "Upper Right Central Incisor Pulp",
    112: "Upper Right Lateral Incisor Pulp",
    113: "Upper Right Canine Pulp",
    114: "Upper Right First Premolar Pulp",
    115: "Upper Right Second Premolar Pulp",
    116: "Upper Right First Molar Pulp",
    117: "Upper Right Second Molar Pulp",
    118: "Upper Right Third Molar Pulp",
    121: "Upper Left Central Incisor Pulp",
    122: "Upper Left Lateral Incisor Pulp",
    123: "Upper Left Canine Pulp",
    124: "Upper Left First Premolar Pulp",
    125: "Upper Left Second Premolar Pulp",
    126: "Upper Left First Molar Pulp",
    127: "Upper Left Second Molar Pulp",
    128: "Upper Left Third Molar Pulp",
    131: "Lower Left Central Incisor Pulp",
    132: "Lower Left Lateral Incisor Pulp",
    133: "Lower Left Canine Pulp",
    134: "Lower Left First Premolar Pulp",
    135: "Lower Left Second Premolar Pulp",
    136: "Lower Left First Molar Pulp",
    137: "Lower Left Second Molar Pulp",
    138: "Lower Left Third Molar Pulp",
    141: "Lower Right Central Incisor Pulp",
    142: "Lower Right Lateral Incisor Pulp",
    143: "Lower Right Canine Pulp",
    144: "Lower Right First Premolar Pulp",
    145: "Lower Right Second Premolar Pulp",
    146: "Lower Right First Molar Pulp",
    147: "Lower Right Second Molar Pulp",
    148: "Lower Right Third Molar Pulp",
}

# Initialize Statistics

In [4]:
stats = {}

for label_id, label_name in LABELS.items():
    stats[label_id] = {
        "name": label_name,
        "cases_present": 0,
        "voxel_counts": []
    }

# Scan All Masks

In [5]:
mask_files = sorted(LABELS_FOLDER.glob("*.nii.gz"))

for mask_file in tqdm(mask_files):

    mask = nib.load(str(mask_file)).get_fdata()

    unique_labels = np.unique(mask)

    for label_id in LABELS:

        voxel_count = np.sum(mask == label_id)

        if voxel_count > 0:
            stats[label_id]["cases_present"] += 1
            stats[label_id]["voxel_counts"].append(voxel_count)

100%|██████████| 532/532 [24:32<00:00,  2.77s/it]


# Create DataFrame

In [6]:
total_cases = len(mask_files)

rows = []

for label_id, info in stats.items():

    voxels = info["voxel_counts"]

    if len(voxels) == 0:
        continue

    rows.append({
        "Label ID": label_id,
        "Label Name": info["name"],
        "Cases Present": info["cases_present"],
        "Presence %":
            round(
                100 * info["cases_present"] / total_cases,
                2
            ),
        "Total Voxels":
            int(np.sum(voxels)),
        "Mean Voxels":
            int(np.mean(voxels)),
        "Median Voxels":
            int(np.median(voxels)),
        "Min Voxels":
            int(np.min(voxels)),
        "Max Voxels":
            int(np.max(voxels))
    })

df = pd.DataFrame(rows)

df = df.sort_values(
    "Cases Present",
    ascending=False
)

df.head()

,Label ID,Label Name,Cases Present,Presence %,Total Voxels,Mean Voxels,Median Voxels,Min Voxels,Max Voxels
0,1,Lower Jawbone,532,100.00,828087394,1556555,1528107,521383,2942737
2,3,Left Inferior Alveolar Canal,532,100.00,7933456,14912,14483,6198,32604
3,4,Right Inferior Alveolar Canal,532,100.00,8048610,15128,14678,5610,30354
6,7,Pharynx,532,100.00,258369432,485656,436694,71533,1503552
44,105,Lingual Canal,519,97.56,154736,298,276,6,835


# Save Excel

In [7]:
output_file = "ToothFairy3_Label_Analysis.xlsx"

df.to_excel(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ToothFairy3_Label_Analysis.xlsx


# Most Common Structures

In [8]:
df[
    ["Label ID",
     "Label Name",
     "Cases Present",
     "Presence %"]
].head(20)

,Label ID,Label Name,Cases Present,Presence %
0,1,Lower Jawbone,532,100.00
2,3,Left Inferior Alveolar Canal,532,100.00
3,4,Right Inferior Alveolar Canal,532,100.00
6,7,Pharynx,532,100.00
44,105,Lingual Canal,519,97.56
42,103,Left Mandibular Incisive Canal,503,94.55
43,104,Right Mandibular Incisive Canal,494,92.86
36,43,Lower Right Canine,493,92.67
71,143,Lower Right Canine Pulp,489,91.92
28,33,Lower Left Canine,486,91.35


# Rare Structures

In [9]:
df[
    ["Label ID",
     "Label Name",
     "Cases Present",
     "Presence %"]
].tail(20)

,Label ID,Label Name,Cases Present,Presence %
19,22,Upper Left Lateral Incisor,209,39.29
11,12,Upper Right Lateral Incisor,196,36.84
8,9,Crown,193,36.28
49,115,Upper Right Second Premolar Pulp,193,36.28
56,124,Upper Left First Premolar Pulp,177,33.27
48,114,Upper Right First Premolar Pulp,163,30.64
25,28,Upper Left Third Molar,159,29.89
17,18,Upper Right Third Molar,147,27.63
47,113,Upper Right Canine Pulp,143,26.88
55,123,Upper Left Canine Pulp,133,25.00


# Category-Level Summary

In [10]:
teeth_labels = [
    x for x in LABELS
    if (
        11 <= x <= 48
        and x not in [19,20,29,30,39,40]
    )
]

pulp_labels = [
    x for x in LABELS
    if 111 <= x <= 148
]

print("Teeth classes :", len(teeth_labels))
print("Pulp classes  :", len(pulp_labels))

Teeth classes : 32
Pulp classes  : 32
